In [1]:
import pandas as pd
import random
from typing import List

N = 1667

In [2]:
df = pd.read_excel("Final question list - 48.xlsx")
df

,Domain,Facet,Index,Question,p1,p2,p3
0,Honesty-Humility,Sincerity,F1Q2,People sometimes act more friendly or flatteri...,People sometimes act more friendly or flatteri...,"Do you tend to act nicer than you feel, or do ...","If it helps, you can share an example that sho..."
1,Honesty-Humility,Sincerity,F1Q3,Sometimes people adjust what they say because ...,Sometimes people adjust what they say because ...,"Do you tend to be straightforward, or adjust y...","If it helps, you can share an example that sho..."
2,Honesty-Humility,Fairness,F2Q1,"Sometimes, people receive something that wasn’...","Sometimes, people receive something that wasn’...",What thoughts or feelings usually guide your c...,"If it helps, you can share an example that sho..."
3,Honesty-Humility,Fairness,F2Q2,There are situations where people can gain an ...,There are situations where people can gain an ...,"Do you tend to follow the rules, or sometimes...","If it helps, you can share an example that re..."
4,Honesty-Humility,Greed-Avoidance,F3Q1,Sometimes people consider buying something exp...,Sometimes people consider buying something exp...,"When you face that kind of choice, what usual...","If it helps, you can share an example that sh..."
5,Honesty-Humility,Greed-Avoidance,F3Q2,Sometimes we find ourselves in settings where ...,Sometimes we find ourselves in settings where ...,"In general, if you are in that kind of enviro...","If it helps, you can share an example that sho..."
6,Honesty-Humility,Modesty,F4Q1,"Sometimes, people are given special privileges...","Sometimes, people are given special privileges...","In general, if you are offered or expected to...","If it helps, you can share an example that sho..."
7,Honesty-Humility,Modesty,F4Q3,"In settings like school, work, or team project...","In settings like school, work, or team project...",How do you usually respond if others don’t not...,"If it helps, you can share an example that ref..."
8,Emotionality,Fearfulness,F5Q1,Sometimes we need to go somewhere despite unsa...,Sometimes we need to go somewhere despite unsa...,"In general, if you face that kind of situatio...","If it helps, you can share an example that sh..."
9,Emotionality,Fearfulness,F5Q2,Sometimes people are invited or encouraged to ...,Sometimes people are invited or encouraged to ...,"In general, if you are invited or encouraged t...","If it helps, you can share an example that re..."


In [3]:
df["Domain"].value_counts()

Domain
Honesty-Humility          8
Emotionality              8
Extraversion              8
Agreeableness             8
Conscientiousness         8
Openness to Experience    8
Name: count, dtype: int64

# generate_question_orders

In [4]:
"""
For each group, generat 1667 list
"""

groups = ["HEX", "EXA", "XAC", "ACO", "COH", "OHE"]
mappings = {
    "H": "Honesty-Humility",
    "E" : "Emotionality",
    "X": "Extraversion",
    "A": "Agreeableness",
    "C": "Conscientiousness",
    "O": "Openness to Experience"
}



def get_question_index(group: str = "HEX"):
    selected_domains = [mappings[g] for g in group] # ['Emotionality', 'Extraversion', 'Agreeableness']

    df_selected = df[df["Domain"].isin(selected_domains)].reset_index(drop=True)
    indices = df_selected["Index"].to_list()
    return indices



def shuffle(indices: List[str]):
    random.shuffle(indices)
    return indices

In [5]:
shuffle(get_question_index("EXA"))

['F5Q2',
 'F16Q3',
 'F10Q2',
 'F15Q1',
 'F14Q1',
 'F5Q1',
 'F7Q2',
 'F10Q1',
 'F12Q1',
 'F6Q3',
 'F8Q3',
 'F9Q1',
 'F6Q1',
 'F11Q1',
 'F13Q2',
 'F12Q2',
 'F15Q3',
 'F8Q2',
 'F11Q2',
 'F7Q1',
 'F16Q1',
 'F9Q2',
 'F13Q1',
 'F14Q2']

In [37]:
for group in groups:
    for i in range(N):
        indices = shuffle(get_question_index(group))

        with open(f"./conversation_engine/question_list/meta/{group}_{i}.txt", "w") as file:
            file.write(str(indices))

In [47]:
asd = " asd  \n"
asd.strip()

'asd'

# generate code tempate

In [33]:
def generate_code(index, question_part1, question_part2, question_part3):
    return f"""
# ---------------------------------------------------------------------------- #
# ------------------------------------ {index} ------------------------------------ #
# ---------------------------------------------------------------------------- #
node_{index}_0 = DialogNode(
    id="{index}_0",
    text="{question_part1}",
    node_type=NodeType.PLAIN_MESSAGE,
    info={{"progress": 0 }},
)

node_{index}_1 = DialogNode(
    id="{index}_1",
    text="{question_part2}",
    node_type=NodeType.PLAIN_MESSAGE,
    info={{"progress": 0}},
)

node_{index}_2 = DialogNode(
    id="{index}_2",
    text="{question_part3}",
    node_type=NodeType.DEFAULT_QUESTION,
    condition_check=lambda chatting_messages : detect(chatting_messages),
    summary_generators={{
        "NEEDS_CLARIFICATION": Signal.SKIP_SUMMARY_SIGNAL,
        "NONSENSE": Signal.SKIP_SUMMARY_SIGNAL,
        "UNINFORMATIVE": Signal.SKIP_SUMMARY_SIGNAL,
        "VALID": lambda chatting_messages : say_thank_you_and_go_next(chatting_messages),
        "DECLINE_CONTINUE": Signal.SKIP_SUMMARY_SIGNAL,
    }},
    parallism_works={{
        "NEEDS_CLARIFICATION" : [generate_follow_up_question_1_NEEDS_CLARIFICATION],
        "NONSENSE": [generate_follow_up_question_1_NONSENSE],
        "UNINFORMATIVE": [generate_follow_up_question_1_UNINFORMATIVE],
        "VALID": [],
        "DECLINE_CONTINUE": [],
    }},
    info={{"progress": 0 }},
)

node_{index}_3 = DialogNode(
    id="{index}_3",
    text="Dummy",
    node_type=NodeType.DEFAULT_QUESTION,
    condition_check=lambda chatting_messages : detect(chatting_messages),
    summary_generators={{
        "NEEDS_CLARIFICATION": Signal.SKIP_SUMMARY_SIGNAL,
        "NONSENSE": Signal.SKIP_SUMMARY_SIGNAL,
        "UNINFORMATIVE": Signal.SKIP_SUMMARY_SIGNAL,
        "VALID": lambda chatting_messages : say_thank_you_and_go_next(chatting_messages),
        "DECLINE_CONTINUE": Signal.SKIP_SUMMARY_SIGNAL,
    }},
    parallism_works={{
        "NEEDS_CLARIFICATION" : [generate_follow_up_question_2_NEEDS_CLARIFICATION],
        "NONSENSE": [generate_follow_up_question_2_NONSENSE],
        "UNINFORMATIVE": [generate_follow_up_question_2_UNINFORMATIVE],
        "VALID": [],
         "DECLINE_CONTINUE": [],
    }},
    info={{"progress": 0 }},
)


node_{index}_4 = DialogNode(
    id="{index}_4",
    text="Dummy",
    node_type=NodeType.NO_CONDITION_CHECK,
    summary_generators={{
        "default": lambda chatting_messages : say_thank_you_and_go_next(chatting_messages),
    }},
    info={{"progress": 0 }},
)

    
"""

In [48]:
codes = """# This file is generated by code_generating_v2.ipynb
from typing import List
from conversation_engine.node import DialogNode, NodeType, Signal, summary_generator_wrapper
from conversation_engine.question_list_utils import detect, say_thank_you_and_go_next, generate_follow_up_question_1_UNINFORMATIVE, generate_follow_up_question_2_UNINFORMATIVE, generate_follow_up_question_1_NEEDS_CLARIFICATION, generate_follow_up_question_2_NEEDS_CLARIFICATION, generate_follow_up_question_1_NONSENSE, generate_follow_up_question_2_NONSENSE

"""  

for i in range(len(df)):
    question_index, question_part1, question_part2, question_part3 =  df.loc[i, "Index"], df.loc[i, "p1"].strip(), df.loc[i, "p2"].strip(), df.loc[i, "p3"].strip()
    code_part = generate_code(question_index, question_part1, question_part2, question_part3)
    codes += code_part


node_mappings = """
node_mappings = {"""

for i in range(len(df)):
    question_index =  df.loc[i, "Index"]
    node_mappings += f"""
    "{question_index}": [node_{question_index}_0, node_{question_index}_1, node_{question_index}_2, node_{question_index}_3, node_{question_index}_4],"""

    # "F1Q1": [node_F1Q2_0, node_F1Q2_1, node_F1Q2_2, node_F1Q2_3, node_F1Q2_4],
node_mappings += """
}
"""

codes += node_mappings

In [49]:
with open("./conversation_engine/question_list/codes/question_nodes.py", "w") as file:
    file.write(codes)